# AB 测试：营销策略效果评估

基于支付宝广告投放 AB 测试数据，使用 Python 完成实验组与对照组的点击率差异检验。

技术栈：Python (pandas, scipy, matplotlib)

## 1. 项目背景

支付宝对部分用户投放了新广告策略，需要判断新策略是否显著提升了广告点击率。数据集包含 264 万用户，分为对照组（不投放）和实验组（投放新策略），核心指标为是否点击广告。

## 2. 数据加载与概览

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']; matplotlib.rcParams['axes.unicode_minus'] = False
from scipy.stats import chi2_contingency

df = pd.read_csv(r'C:\Users\86186\Desktop\audience_expansion\effect_tb.csv',
                 header=None, names=['dmp_id','user_id','click','flag'])

n1 = (df['dmp_id']==1).sum(); n2 = (df['dmp_id']==2).sum()
c1 = (df[df['dmp_id']==1]['click']==1).sum(); c2 = (df[df['dmp_id']==2]['click']==1).sum()
r1 = c1/n1; r2 = c2/n2

print(f'对照组: {n1:,} 人, 点击率 {r1*100:.2f}%')
print(f'实验组: {n2:,} 人, 点击率 {r2*100:.2f}%')
print(f'相对提升: {(r2/r1-1)*100:.1f}%')


## 3. 卡方检验

检验两组点击率差异是否统计显著。

In [ ]:
ct = pd.crosstab(df['dmp_id'], df['click'])
chi2, p, dof, expected = chi2_contingency(ct)
print(ct)
print(f'\nchi2 = {chi2:.2f}, p = {p:.2e}')
print('=> 差异极显著 (p < 0.001)，实验组策略确实提升了点击率')


## 4. 效应量与置信区间

统计显著不等于业务显著。计算效应量和 95% 置信区间来判断提升幅度是否值得上线。

In [ ]:
diff = r2 - r1
se = np.sqrt(r1*(1-r1)/n1 + r2*(1-r2)/n2)
ci_l, ci_u = diff - 1.96*se, diff + 1.96*se

print(f'点击率差异: {diff*100:.4f} 个百分点')
print(f'95% CI: [{ci_l*100:.4f}%, {ci_u*100:.4f}%]')
print(f'结论: 区间远高于0，效应可靠且幅度有业务价值')


## 5. 可视化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

rates = [r1*100, r2*100]
bars = ax1.bar(['对照组','实验组'], rates, color=['steelblue','darkorange'], width=0.4)
for b, r in zip(bars, rates): ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{r:.2f}%', ha='center', fontsize=14)
ax1.set_ylabel('点击率 (%)'); ax1.set_title('点击率对比')

ax2.errorbar(['差异'], [diff*100], yerr=[[(diff-ci_l)*100],[(ci_u-diff)*100]], fmt='o', color='darkorange', capsize=10, markersize=10, linewidth=2)
ax2.axhline(y=0, color='gray', linestyle='--')
ax2.set_ylabel('点击率差异 (百分点)'); ax2.set_title(f'效应量 (p={p:.2e})')
ax2.set_ylim(0, (ci_u*100)*1.5)
ax2.annotate(f'{diff*100:.4f}%', xy=(0, diff*100), xytext=(0.15, diff*100+0.02), fontsize=12)
plt.tight_layout(); plt.show()


## 6. 结论与建议

**统计结论：** 实验组点击率 (1.68%) 显著高于对照组 (1.28%)，p < 0.001，95% CI 不含 0。卡方值 749.69，效应可靠。

**业务结论：** 新策略带来 31.8% 的相对提升和 0.41 个百分点的绝对提升。虽然绝对提升幅度小（广告点击本身就是低概率事件），但在 264 万样本下差异稳定且显著。

**建议：** 全量上线实验组投放策略。需要额外关注：
- 策略是否对不同用户群体（新老用户、活跃度）效果不同
- 点击率提升是否转化为最终转化（如下载、购买）的提升
- 投放成本的增加是否在 ROI 可接受范围内

## 7. 技术栈

Python (pandas, scipy, matplotlib) · 卡方检验 · 效应量 · 置信区间 · AB 测试